#Silver Layer

In [0]:
trips_clean = spark.table("workspace.silver.trips_clean")
dim_zone = spark.table("workspace.silver.dim_zone")

#Auditing the imports from silver

In [0]:
trips_clean.show()

+-------------+-------------------+-------------------+-----------+--------------+-------------+-------------+-----------+----------------------+---------+--------------+-----------+------------------+----------+-----------------------+------------+-------------------+-----------+---------------------+--------------------+------------+---------+------------------+------------------+--------------+---------+------------------+------------------+---------------+---------------+---------------+------------------+------------------+-------------+-------------+-------------------+-----------------+
|provider_code|   pickup_timestamp|  dropoff_timestamp|rider_count|distance_miles|rate_class_id|origin_loc_id|dest_loc_id|fare_settlement_method|base_fare|surcharge_misc|transit_tax|driver_tip_payment|toll_total|service_improvement_fee|charge_total|zone_congestion_fee|Airport_fee|congestion_relief_fee|         source_file|source_month|pu_loc_id|        pickup_lat|        pickup_lon|pickup_borough

#Auditing the imported Data

In [0]:
from pyspark.sql import functions as F

null_counts = trips_clean.select([
    F.round((F.count(F.when(F.col(c).isNull(), c)) / F.count(F.lit(1))) * 100, 2).alias(c)
    for c in trips_clean.columns
])
null_counts.show(vertical=True)

-RECORD 0-----------------------
 provider_code           | 0.0  
 pickup_timestamp        | 0.0  
 dropoff_timestamp       | 0.0  
 rider_count             | 0.0  
 distance_miles          | 0.0  
 rate_class_id           | 0.0  
 origin_loc_id           | 0.0  
 dest_loc_id             | 0.0  
 fare_settlement_method  | 0.0  
 base_fare               | 0.0  
 surcharge_misc          | 0.0  
 transit_tax             | 0.0  
 driver_tip_payment      | 0.0  
 toll_total              | 0.0  
 service_improvement_fee | 0.0  
 charge_total            | 0.0  
 zone_congestion_fee     | 0.0  
 Airport_fee             | 0.0  
 congestion_relief_fee   | 0.0  
 source_file             | 0.0  
 source_month            | 0.0  
 pu_loc_id               | 0.0  
 pickup_lat              | 0.19 
 pickup_lon              | 0.19 
 pickup_borough          | 0.0  
 do_loc_id               | 0.0  
 dropoff_lat             | 0.66 
 dropoff_lon             | 0.66 
 dropoff_borough         | 0.0  
 pickup_h3

##Null handling

In [0]:
trips_stage1 = trips_clean.dropna(subset=[
    "origin_loc_id", "dest_loc_id", "pickup_timestamp",
    "dropoff_timestamp", "distance_miles"
])

In [0]:
trips_stage1.count()

48601782

In [0]:
trips_stage2 = trips_stage1.withColumn(
    "has_valid_geo",
    F.col("pickup_lat").isNotNull() & F.col("dropoff_lat").isNotNull()
).withColumn(
    "pickup_borough", F.coalesce(F.col("pickup_borough"), F.lit("Unknown"))
).withColumn(
    "dropoff_borough", F.coalesce(F.col("dropoff_borough"), F.lit("Unknown"))
)

In [0]:
trips_stage2.show()

+-------------+-------------------+-------------------+-----------+--------------+-------------+-------------+-----------+----------------------+---------+--------------+-----------+------------------+----------+-----------------------+------------+-------------------+-----------+---------------------+--------------------+------------+---------+------------------+------------------+--------------+---------+------------------+------------------+---------------+---------------+---------------+------------------+--------------------+-------------+-------------+-------------------+-----------------+
|provider_code|   pickup_timestamp|  dropoff_timestamp|rider_count|distance_miles|rate_class_id|origin_loc_id|dest_loc_id|fare_settlement_method|base_fare|surcharge_misc|transit_tax|driver_tip_payment|toll_total|service_improvement_fee|charge_total|zone_congestion_fee|Airport_fee|congestion_relief_fee|         source_file|source_month|pu_loc_id|        pickup_lat|        pickup_lon|pickup_borou

In [0]:
count1 = trips_stage1.count()
count2 = trips_stage2.count()
print(f"Removed Nulls: {count1 - count2}")

Removed Nulls: 0


In [0]:
fee_cols = ["surcharge_misc", "transit_tax", "toll_total",
            "service_improvement_fee", "zone_congestion_fee",
            "Airport_fee", "congestion_relief_fee", "driver_tip_payment"]

trips_stage3 = trips_stage2.fillna(0.0, subset=fee_cols)

In [0]:
from pyspark.sql import functions as F

null_counts_after_ts3 = trips_stage3.select([
    F.round((F.count(F.when(F.col(c).isNull(), c)) / F.count(F.lit(1))) * 100, 2).alias(c)
    for c in trips_stage3.columns
])
null_counts_after_ts3.show(vertical=True)

-RECORD 0-----------------------
 provider_code           | 0.0  
 pickup_timestamp        | 0.0  
 dropoff_timestamp       | 0.0  
 rider_count             | 0.0  
 distance_miles          | 0.0  
 rate_class_id           | 0.0  
 origin_loc_id           | 0.0  
 dest_loc_id             | 0.0  
 fare_settlement_method  | 0.0  
 base_fare               | 0.0  
 surcharge_misc          | 0.0  
 transit_tax             | 0.0  
 driver_tip_payment      | 0.0  
 toll_total              | 0.0  
 service_improvement_fee | 0.0  
 charge_total            | 0.0  
 zone_congestion_fee     | 0.0  
 Airport_fee             | 0.0  
 congestion_relief_fee   | 0.0  
 source_file             | 0.0  
 source_month            | 0.0  
 pu_loc_id               | 0.0  
 pickup_lat              | 0.19 
 pickup_lon              | 0.19 
 pickup_borough          | 0.0  
 do_loc_id               | 0.0  
 dropoff_lat             | 0.66 
 dropoff_lon             | 0.66 
 dropoff_borough         | 0.0  
 pickup_h3

###Imputing

rider_count: imputed by 1 (Assuming one ride has one passenger only)

rate_class_id: imputed by mode

offline_record_flag: N

In [0]:
trips_stage4 = trips_stage3 \
    .withColumn("rider_count_imputed", F.col("rider_count").isNull()) \
    .withColumn("rider_count", F.coalesce(F.col("rider_count"), F.lit(1.0))) \
    .withColumn("offline_record_flag", F.lit("N"))

In [0]:
from pyspark.sql import functions as F

null_counts_after_ts3 = trips_stage4.select([
    F.round((F.count(F.when(F.col(c).isNull(), c)) / F.count(F.lit(1))) * 100, 2).alias(c)
    for c in trips_stage4.columns
])
null_counts_after_ts3.show(vertical=True)

-RECORD 0-----------------------
 provider_code           | 0.0  
 pickup_timestamp        | 0.0  
 dropoff_timestamp       | 0.0  
 rider_count             | 0.0  
 distance_miles          | 0.0  
 rate_class_id           | 0.0  
 origin_loc_id           | 0.0  
 dest_loc_id             | 0.0  
 fare_settlement_method  | 0.0  
 base_fare               | 0.0  
 surcharge_misc          | 0.0  
 transit_tax             | 0.0  
 driver_tip_payment      | 0.0  
 toll_total              | 0.0  
 service_improvement_fee | 0.0  
 charge_total            | 0.0  
 zone_congestion_fee     | 0.0  
 Airport_fee             | 0.0  
 congestion_relief_fee   | 0.0  
 source_file             | 0.0  
 source_month            | 0.0  
 pu_loc_id               | 0.0  
 pickup_lat              | 0.19 
 pickup_lon              | 0.19 
 pickup_borough          | 0.0  
 do_loc_id               | 0.0  
 dropoff_lat             | 0.66 
 dropoff_lon             | 0.66 
 dropoff_borough         | 0.0  
 pickup_h3

####Maintaining the data quality among params

In [0]:
trips_final = trips_stage4.withColumn(
    "is_valid_trip",
    (F.col("trip_duration_min").between(1, 240)) &          
    (F.col("distance_miles").between(0.1, 100)) &            
    (F.coalesce(F.col("avg_speed_mph"), F.lit(0)) <= 100) &   #Expressway speed
    F.col("pickup_lat").isNotNull() &
    F.col("dropoff_lat").isNotNull() &
    (F.col("pickup_timestamp").cast("date").between("2025-04-01", "2026-03-31"))
)

In [0]:
from pyspark.sql import functions as F

null_counts_after_ts4 = trips_final.select([
    F.round((F.count(F.when(F.col(c).isNull(), c)) / F.count(F.lit(1))) * 100, 2).alias(c)
    for c in trips_final.columns
])
null_counts_after_ts4.show(vertical=True)

-RECORD 0-----------------------
 provider_code           | 0.0  
 pickup_timestamp        | 0.0  
 dropoff_timestamp       | 0.0  
 rider_count             | 0.0  
 distance_miles          | 0.0  
 rate_class_id           | 0.0  
 origin_loc_id           | 0.0  
 dest_loc_id             | 0.0  
 fare_settlement_method  | 0.0  
 base_fare               | 0.0  
 surcharge_misc          | 0.0  
 transit_tax             | 0.0  
 driver_tip_payment      | 0.0  
 toll_total              | 0.0  
 service_improvement_fee | 0.0  
 charge_total            | 0.0  
 zone_congestion_fee     | 0.0  
 Airport_fee             | 0.0  
 congestion_relief_fee   | 0.0  
 source_file             | 0.0  
 source_month            | 0.0  
 pu_loc_id               | 0.0  
 pickup_lat              | 0.19 
 pickup_lon              | 0.19 
 pickup_borough          | 0.0  
 do_loc_id               | 0.0  
 dropoff_lat             | 0.66 
 dropoff_lon             | 0.66 
 dropoff_borough         | 0.0  
 pickup_h3

In [0]:
trips_final = trips_final.withColumn(
    "rate_class_id",
    F.coalesce(F.col("rate_class_id"), F.lit(-1))  #-1 stands for unknown data
)

In [0]:
trips_final.show()

+-------------+-------------------+-------------------+-----------+--------------+-------------+-------------+-----------+----------------------+---------+--------------+-----------+------------------+----------+-----------------------+------------+-------------------+-----------+---------------------+--------------------+------------+---------+------------------+------------------+--------------+---------+------------------+------------------+---------------+---------------+---------------+------------------+--------------------+-------------+-------------+-------------------+-----------------+-------------------+
|provider_code|   pickup_timestamp|  dropoff_timestamp|rider_count|distance_miles|rate_class_id|origin_loc_id|dest_loc_id|fare_settlement_method|base_fare|surcharge_misc|transit_tax|driver_tip_payment|toll_total|service_improvement_fee|charge_total|zone_congestion_fee|Airport_fee|congestion_relief_fee|         source_file|source_month|pu_loc_id|        pickup_lat|        pic

In [0]:
unique_count = trips_final.select("offline_record_flag").distinct().count()
print(unique_count)

1


In [0]:
trips_final = trips_final.withColumn(
    "is_offline_record",
    F.col("offline_record_flag") == "Y"
)

In [0]:
trips_final = trips_final.drop("offline_record_flag")

In [0]:
trips_final.show()

+-------------+-------------------+-------------------+-----------+--------------+-------------+-------------+-----------+----------------------+---------+--------------+-----------+------------------+----------+-----------------------+------------+-------------------+-----------+---------------------+--------------------+------------+---------+------------------+------------------+--------------+---------+------------------+------------------+---------------+---------------+---------------+------------------+--------------------+-------------+-------------+-------------------+-----------------+
|provider_code|   pickup_timestamp|  dropoff_timestamp|rider_count|distance_miles|rate_class_id|origin_loc_id|dest_loc_id|fare_settlement_method|base_fare|surcharge_misc|transit_tax|driver_tip_payment|toll_total|service_improvement_fee|charge_total|zone_congestion_fee|Airport_fee|congestion_relief_fee|         source_file|source_month|pu_loc_id|        pickup_lat|        pickup_lon|pickup_borou

In [0]:
geo_valid_trips = trips_final.filter(F.col("has_valid_geo") == True)

all_trips = trips_final #backing up

In [0]:
(trips_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("source_month")
    .saveAsTable("workspace.silver.trips_clean")
)

In [0]:
%sql
SELECT COUNT(provider_code) FROM silver.trips_clean

COUNT(provider_code)
48601782


In [0]:
trips_clean.filter(~F.col("is_valid_trip")).agg(
    F.count("*").alias("total_invalid_trips")
).show()

trips_clean.filter(F.col("is_valid_trip")).agg(
    F.min(F.col("pickup_timestamp").cast("date")).alias("min_date"),
    F.max(F.col("pickup_timestamp").cast("date")).alias("max_date")
).show()

+-------------------+
|total_invalid_trips|
+-------------------+
|            2908613|
+-------------------+

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-04-01|2026-03-31|
+----------+----------+

